# Pythia8 + ROOT Hands-On exercise  
### By Oliver Matonoha  

In this Jupyter notebook, we will:  
- Use Pythia8 to simulate high-energy collisions and generate events for RHIC and LHC energies  
- Analyze specific particles in given decay channels such as \( J/\psi \to \mu^+ \mu^- \).  
- Explore Multiple Partonic Interactions (MPI).    
- Use **ROOT** to visualize results, create histograms, and store data in ROOT trees for further analysis.  

You can access the Python interface documentation of Pythia8 using:  
```python
help(pythia8)


First, we load the libraries:

In [ ]:
import pythia8
import ROOT

# For interactive canvas, uncomment the following line
#%jsroot on

# Displays the help for the pythia8 module
help(pythia8)

### 1. Multiplicity and MPI in events at 7 TeV

We initialize **proton-proton (pp) collisions at √s = 7 TeV** using **Pythia8** with **soft QCD processes** (MB). The key steps:

- `Beams:idA = 2212`, `Beams:idB = 2212`, `Beams:eCM = 7000` -- sets up beams and energy
- `SoftQCD:nonDiffractive = on` selects events mostly corresponding to minimum bias.
- Remember to change when simulating actual datasets! `Random:seed = 0` will use a unique seed based on the current time.
- `pythia.init()` estimates cross-sections and prepares the simulation.

Full list of configuration settings: [Online Pythia8 Manual](https://pythia.org//latest-manual/Welcome.html)

- `pythia.next()` produces the next event.
- *Selecting final-state particles:*
  - `particle.isFinal()`: Take only final state particles
  - `particle.isCharged()`: Restricts to charged tracks.
  - `|η| < 0.5`: Focuses on **mid-rapidity**, relevant for LHC central detectors.
- `pythia.stat()` prints cross-sections and process details

Properties of particles within events are documented at [Particle Class](https://pythia.org/latest-manual/ParticleProperties.html)

Furthermore, we want to study the parameter related to MPI definition:
- `MultipartonInteractions:pT0Ref = 3.0` → Controls the minimum pT scale for MPI.

We will look at:
1. Number of Multiple Parton Interactions (MPI) per event.
2. Correlation between multiplicity and MPI (2D histogram).
3. Comparison with experimental data from **CMS** (https://www.hepdata.net/record/57914)

The multiplicity histogram is normalized for direct data comparison.  
A log-scale y-axis is applied to better visualize distribution tails.


In [ ]:
# Initialize Pythia
pythia = pythia8.Pythia()

# Set up LHC proton-proton collisions at 7 TeV
pythia.readString("Beams:idA = 2212")  # Proton
pythia.readString("Beams:idB = 2212")  # Proton
pythia.readString("Beams:eCM = 7000.")  # Collision energy in GeV

# Enable SoftQCD Non-Single Diffractive (NSD) events
pythia.readString("SoftQCD:nonDiffractive = on")

# Minimum pT for multiple parton interactions (user can modify)
pythia.readString("MultipartonInteractions:pT0Ref = 3.0")

# Enable MPI, ISR, FSR
pythia.readString("PartonLevel:MPI = on")
pythia.readString("PartonLevel:ISR = on")
pythia.readString("PartonLevel:FSR = on")

# Enable hadron decays
pythia.readString("HadronLevel:Decay = on")

# Initialize Pythia
pythia.init()

# ROOT histogram setup
canvas = ROOT.TCanvas("c2", "Multiplicity, n_MPI, and 2D Histogram", 900, 600)

# Create histograms
h_multiplicity = ROOT.TH1F("h_multiplicity", "Charged Primary Multiplicity;N_{ch};Normalized Events", 50, 0, 50)
h_nMPI = ROOT.TH1F("h_nMPI", "Number of MPI;N_{MPI};Events", 25, 0, 25)
h_2D = ROOT.TH2F("h_2D", "Multiplicity vs. Number of MPI;N_{MPI};N_{ch}", 25, 0, 25, 50, 0, 50)

# Event loop
nEvents = 2000
for iEvent in range(nEvents):
    if not pythia.next():
        continue
    
    # Get number of multiple parton interactions (MPI)
    nMPI = pythia.infoPython().nMPI()
    h_nMPI.Fill(nMPI)
    
    # Count charged primary final-state particles
    nCharged = 0
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        if p.isFinal() and p.isCharged() and abs(p.eta()) < 0.5:
            nCharged += 1
    
    h_multiplicity.Fill(nCharged)
    h_2D.Fill(nMPI, nCharged)

# Normalize multiplicity histogram
h_multiplicity.Scale(1.0 / nEvents)

# Load experimental data from ROOT file
exp_file = ROOT.TFile("HEPData-ins879315-v1-Table_12.root")
data_dir = exp_file.Get("Table 12")
graph_exp = data_dir.Get("Graph1D_y1")

# Draw histograms
canvas.Divide(2, 2)
canvas.cd(1)
canvas.GetPad(1).SetLogy()  # Set log scale for y-axis
h_multiplicity.Draw("HIST")
graph_exp.SetMarkerStyle(20)
graph_exp.SetMarkerColor(ROOT.kRed)
graph_exp.Draw("P SAME")

canvas.cd(2)
h_nMPI.Draw()
canvas.cd(3)
h_2D.Draw("COLZ")
canvas.Draw()

# Save histograms to ROOT file
output_file = ROOT.TFile("lhc_mpi_analysis.root", "RECREATE")
h_multiplicity.Write()
h_nMPI.Write()
h_2D.Write()
output_file.Close()

# Print final statistics
pythia.stat()

print("\nSimulation complete. ROOT file 'lhc_mpi_analysis.root' saved with histograms.")


Now we simulated 2000 pp collision events at 7 TeV. What happened??

1. *Comparison of `N_ch` and data*
   - Something needs to be tuned!

2. *Number of MPI (`N_MPI`)*
   - Number of **multiple parton interactions** per event.

3. *Multiplicity vs. MPI (2D Histogram)*
   - Shows correlation between `N_ch` and `N_MPI`.
   - Higher MPI is the dominant source of higher multiplicity

#### **Play with the code!**
- Add a pT cut > 0.15 GeV/c (to be more realistic)
- What happens to `N_ch` and `N_MPI` as we change the MPI minimum pT parameter pT0Ref?
- What value of pT0Ref is optimal to reproduce the CMS data?

### 2. J/ψ production in pp Collisions

In this exercise, we simulate J/ψ production in proton-proton collisions at 7 TeV and store key event and particle information in a structured ROOT tree.

What happens in the code?
- *Accessing PDG IDs*:  
  - We identify the J/ψ using its `PDG ID = 443`.
  - Muons are selected by comparing their `PDG ID = 13`.
  - Decay modes of the particles can be turned on/off. We select only the di-muon channel `443:onIfMatch = 13 -13`

- *Tracking ancestry*:
  - Each muon’s mother index is stored (`mu_motherID`).
  - The PDG ID of the mother is also stored (`mu_motherPDG`).

- *Tree Structure*:
  - `nMPI` and `nCh`
  - J/ψ `pT`, `y`, `φ`, and its index in the event (`jpsi_id`).
  - Muon `pT`, `η`, `φ`, charge, mother index, and mother PDG ID.


In [ ]:
# Initialize Pythia
pythia = pythia8.Pythia()

# Set up LHC proton-proton collisions at 7 TeV
pythia.readString("Beams:idA = 2212")  # Proton
pythia.readString("Beams:idB = 2212")  # Proton
pythia.readString("Beams:eCM = 7000.")  # Collision energy in GeV

# Enable J/psi production
pythia.readString("Charmonium:all = on")

# Ensure J/Psi is decaying only into dimuons
pythia.readString("443:onMode = off")       # Turn off all J/Psi decays
pythia.readString("443:onIfMatch = 13 -13") # Enable only J/Psi -> mu+ mu-


# Enable MPI, ISR, FSR
pythia.readString("PartonLevel:MPI = on")
pythia.readString("PartonLevel:ISR = on")
pythia.readString("PartonLevel:FSR = on")

# Enable hadron decays
pythia.readString("HadronLevel:Decay = on")

# Initialize Pythia
pythia.init()

# Create a ROOT file and tree
output_file = ROOT.TFile("jpsi_production.root", "RECREATE")
tree = ROOT.TTree("T", "J/psi Production Data")

# Define event-level variables
nMPI = ROOT.std.vector('int')(1)
nCh = ROOT.std.vector('int')(1)

# Define J/psi mother variables
jpsi_pT = ROOT.std.vector('float')()
jpsi_y = ROOT.std.vector('float')()
jpsi_phi = ROOT.std.vector('float')()
jpsi_id = ROOT.std.vector('int')()

# Define muon daughter variables
mu_pT = ROOT.std.vector('float')()
mu_eta = ROOT.std.vector('float')()
mu_phi = ROOT.std.vector('float')()
mu_charge = ROOT.std.vector('int')()
mu_motherID = ROOT.std.vector('int')()
mu_motherPDG = ROOT.std.vector('int')()

# Attach branches to the tree
tree.Branch("nMPI", nMPI)
tree.Branch("nCh", nCh)
tree.Branch("jpsi_pT", jpsi_pT)
tree.Branch("jpsi_y", jpsi_y)
tree.Branch("jpsi_phi", jpsi_phi)
tree.Branch("jpsi_id", jpsi_id)
tree.Branch("mu_pT", mu_pT)
tree.Branch("mu_eta", mu_eta)
tree.Branch("mu_phi", mu_phi)
tree.Branch("mu_charge", mu_charge)
tree.Branch("mu_motherID", mu_motherID)
tree.Branch("mu_motherPDG", mu_motherPDG)

# Event loop
nEvents = 5000
for iEvent in range(nEvents):
    if not pythia.next():
        continue
    
    # Clear vectors for new event
    nMPI[0] = pythia.infoPython().nMPI()
    
    nCharged = 0
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        if p.isFinal() and p.isCharged() and abs(p.eta()) < 2.5 and p.pT() > 0.15:
            nCharged += 1
    nCh[0] = nCharged
    
    jpsi_pT.clear()
    jpsi_y.clear()
    jpsi_phi.clear()
    jpsi_id.clear()
    mu_pT.clear()
    mu_eta.clear()
    mu_phi.clear()
    mu_charge.clear()
    mu_motherID.clear()
    mu_motherPDG.clear()
    
    # Loop over particles to find J/psi and its decay products
    for i in range(pythia.event.size()):
        p = pythia.event[i]
        
        if abs(p.id()) == 443:  # J/psi identification
            jpsi_pT.push_back(p.pT())
            jpsi_y.push_back(p.y())
            jpsi_phi.push_back(p.phi())
            jpsi_id.push_back(i)
        
        # Find muon daughters of J/psi
        if abs(p.id()) == 13 and p.isFinal():  # Muon or anti-muon
            mu_pT.push_back(p.pT())
            mu_eta.push_back(p.eta())
            mu_phi.push_back(p.phi())
            mu_charge.push_back(int(p.id() / abs(p.id())))  # +1 for mu+, -1 for mu-
            mu_motherID.push_back(p.mother1())
            mu_motherPDG.push_back(pythia.event[p.mother1()].id())
    
    tree.Fill()

# Save tree and close file
output_file.Write()
output_file.Close()

# Print final statistics
pythia.stat()

print("\nSimulation complete. ROOT file 'jpsi_production.root' saved with J/psi data.")


Now we can browse the TTree in ROOT!

1) What is the average pT of the muons that come from Jpsi decays?

---

We can generate the same trees using a python script prepared in this repository:
```
python simulate_jpsi.py
```

Or using a C++ script for much better performance (also prepared, needs to be compiled first):
```
g++ -o simulate_jpsi simulate_jpsi.cpp `root-config --cflags --libs` -I$PYTHIA8/include -L$PYTHIA8/lib -lpythia8  -Wl,-rpath,$PYTHIA8/lib
./simulate_jpsi
```

Try both and compare the performance!


### 3. Heavy-ion collisions in PYTHIA: “Hello Angantyr” (O+O at 200 GeV)

So far we simulated **pp** events, where the “activity” in the event is mainly driven by things like **MPI** and showers.

For **A+A** collisions we need something additional -- geometry.
Angantyr is PYTHIA’s heavy-ion model which builds an A+A event from many underlying nucleon–nucleon sub-collisions sampled using Glauber.

For nucleus–nucleus collisions we additionally need to tell Pythia that we are in **heavy-ion mode**.

In `angantyr.cpp`, the important settings are:

- `Beams:idA = 1000080160`  
  `Beams:idB = 1000080160`  
  → oxygen-16 nuclei (PDG ion code format)

- `Beams:eCM = 200.`  
  → √s<sub>NN</sub> = 200 GeV

- `HeavyIon:mode = 1`  
  → activates the Angantyr heavy-ion model

After calling `pythia.init()`, Angantyr internally:

- samples the collision geometry using a Glauber model
- determines the number of wounded nucleons
- generates multiple underlying nucleon–nucleon sub-collisions

Geometry information is accessed in C++ through:
`pythia.info.hiInfo`


Compile and run the script manually in your shell:
```
g++ -O2 -o angantyr angantyr.cpp root-config --cflags --libs -lpythia8 -Wl,-rpath,$PYTHIA8/lib
./angantyr
```

This will generate 20k O+O events. After the simulation is complete, run the following code cell to inspect the outputs.

In [ ]:
# Open the ROOT file produced by angantyr.cpp
f = ROOT.TFile.Open("angantyr.root")

h_nW = f.Get("Nwounded")
h_mult_all = f.Get("MultAll")
h_mult_c0 = f.Get("MultCL0")   # first centrality bin (most central)
h_mult_c2 = f.Get("MultCL2")   # fourth centrality bin (more peripheral)

canvas = ROOT.TCanvas("c_ang", "Angantyr results", 1000, 800)
canvas.Divide(2, 2)

# 1) Nwounded
canvas.cd(1)
canvas.GetPad(1).SetLogy()
h_nW.SetTitle("O+O at #sqrt{s_{NN}} = 200 GeV;N_{wounded};Events")
h_nW.Draw("HIST")

# 2) All-event multiplicity
canvas.cd(2)
canvas.GetPad(2).SetLogy()
h_mult_all.SetTitle("Charged multiplicity (all events);N_{ch};Events")
h_mult_all.Draw("HIST")

# 3) Most central bin
canvas.cd(3)
h_mult_c0.SetTitle("Charged multiplicity (central bin 0-5%);N_{ch};Events")
h_mult_c0.Draw("HIST")

# 4) More peripheral bin
canvas.cd(4)
h_mult_c2.SetTitle("Charged multiplicity (centrality bin 10-20%);N_{ch};Events")
h_mult_c2.Draw("HIST")

canvas.Draw()